## 1. Perbandingan Geographically Weighted Regression, Decision Tree, dan K-Nearest Neighbors dalam Prediksi Pendapatan per Kapita di Chicago

| Komponen | Variabel yang dipakai |
|---|---|
| **Y** | Pendapatan per kapita (`PerCInc14`). |
| **X₁** | Tingkat pengangguran (`Unemp14 / LaborFrc`), yang mewakili kondisi pasar kerja. |
| **X₂** | Proporsi orang dewasa tanpa ijazah SMA (`NoHS14` dibagi total lima kategori pendidikan dari `NoHS14` sampai `ClgGrad14`, termasuk kategori gelar pascasarjana). Ini mewakili tingkat pendidikan wilayah. |
| **Lokasi** | Titik pusat tiap *community area*, dihitung dari geometri wilayah untuk pembobotan GWR. |

dataset : https://geodacenter.github.io/data-and-lab/comarea_vars

## 2. Load Dataset

In [1]:
import geopandas as gpd

gdf = gpd.read_file("dataset/ComArea_ACS14_f.shp")
assert len(gdf) == 77, len(gdf)
assert {"ComAreaID", "community", "PerCInc14", "Unemp14", "LaborFrc",
        "NoHS14", "HSGrad14", "SmClg14", "ClgGrad14"} <= set(gdf.columns), sorted(gdf.columns)
gdf = gdf.to_crs(32616)
gdf[["ComAreaID", "community", "PerCInc14", "Unemp14", "LaborFrc"]].head()

,ComAreaID,community,PerCInc14,Unemp14,LaborFrc
0,35,DOUGLAS,215254,1620,8869
1,36,OAKLAND,54204,865,3139
2,37,FULLER PARK,32828,378,1036
3,38,GRAND BOULEVARD,293535,2619,10756
4,39,KENWOOD,261582,1304,9283


## 3. Feature Engineering

Catatan: spesifikasi menyebut lima kategori pendidikan; dataset ACS ini hanya memuat empat (NoHS14, HSGrad14, SmClg14, ClgGrad14), jadi X2 memakai jumlah keempatnya.

In [2]:
df = gdf.copy()
df["x1_unemp_rate"] = df["Unemp14"] / df["LaborFrc"]
df["x2_no_hs"] = df["NoHS14"] / df[["NoHS14", "HSGrad14", "SmClg14", "ClgGrad14"]].sum(axis=1)
df["y"] = df["PerCInc14"]
centroids = df.geometry.centroid
df["x_m"] = centroids.x
df["y_m"] = centroids.y

assert df[["x1_unemp_rate", "x2_no_hs", "y", "x_m", "y_m"]].isna().sum().sum() == 0
assert df["x1_unemp_rate"].between(0, 1).all()
assert df["x2_no_hs"].between(0, 1).all()
df[["community", "y", "x1_unemp_rate", "x2_no_hs", "x_m", "y_m"]].describe()

,y,x1_unemp_rate,x2_no_hs,x_m,y_m
count,7.700000e+01,77.000000,77.000000,77.000000,7.700000e+01
mean,2.895573e+05,0.158927,0.192422,443958.997676,4.632167e+06
std,3.600327e+05,0.084881,0.114642,5787.984034,1.106758e+04
min,1.490500e+04,0.042418,0.017937,425957.828150,4.612218e+06
25%,1.130910e+05,0.093559,0.108831,440201.764983,4.623674e+06
50%,1.903820e+05,0.140004,0.168188,444506.602087,4.631112e+06
75%,2.861380e+05,0.222155,0.265260,448510.533181,4.641983e+06
max,2.075859e+06,0.404459,0.545832,455498.117941,4.651062e+06


## 4. Train/Test Split

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES = ["x1_unemp_rate", "x2_no_hs"]
idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42)
X_train, X_test = df.loc[idx_train, FEATURES].to_numpy(), df.loc[idx_test, FEATURES].to_numpy()
y_train, y_test = df.loc[idx_train, "y"].to_numpy(), df.loc[idx_test, "y"].to_numpy()
coords_train = df.loc[idx_train, ["x_m", "y_m"]].to_numpy()
coords_test = df.loc[idx_test, ["x_m", "y_m"]].to_numpy()

def evaluate(name, y_true, y_pred):
    y_pred = np.asarray(y_pred).ravel()
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAE": mean_absolute_error(y_true, y_pred)}

results = {}
assert len(idx_train) == 61 and len(idx_test) == 16
assert X_train.shape == (61, 2) and X_test.shape == (16, 2)
print(idx_test.sort_values().tolist())

[0, 4, 10, 12, 18, 28, 33, 35, 45, 47, 50, 53, 58, 66, 68, 74]
